In [19]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
import pickle
import saida
import gerarnovapop
import buscalocal
import matplotlib.pyplot as plt

In [20]:
# -------------------------
# Parâmetros ajustáveis
# -------------------------
NFP_INIT = 5
ALFA = 0.01
NEPOCA = 5

# Parâmetros AG / training
TAM_POP = 50
NUM_GERACOES = 300
TAXA_CRUZA = 0.9
TAXA_MUTA = 0.08
NFP_MAX = 5
FILE_XT = 'xt_inflow.csv'
FILE_YT = 'yt_inflow.csv'

# Early stopping
PATIENCE = 301          # número de gerações sem melhoria para parar
MIN_DELTA = 0.00005         # melhoria mínima considerada relevante

start_time = time.time()

# -------------------------
# Carregar dados
# -------------------------
_raw_X = np.loadtxt(FILE_XT, delimiter=',', skiprows=1)
xt_all = _raw_X[:, 1:]
_raw_y = np.loadtxt(FILE_YT, delimiter=',', skiprows=1)
yt_all = _raw_y[:, 1:].ravel()

npt_total, nin = xt_all.shape

# -------------------------
# Partição: 60% treino / 40% validação (Teste desativado)
# -------------------------
npt_tr = int(round(npt_total * 0.6))

indices = np.arange(len(xt_all))
xt_all = xt_all[indices]
yt_all = yt_all[indices]

xt = xt_all[:npt_tr, :].copy()       # treino (60%)
ydt = yt_all[:npt_tr].copy()         # y de treino

xv = xt_all[npt_tr:, :].copy()  # validação (xv)
ydv = yt_all[npt_tr:].copy()    # y validação (ydv)

npt = npt_tr

# -------------------------
# limites das features
# -------------------------
xmin = xt.min(axis=0)
xmax = xt.max(axis=0)
delta = (xmax - xmin) / (NFP_INIT - 1)

# -------------------------
# função gaussiana
# -------------------------
def gaussmf(x, mean, sigma):
    sigma = np.maximum(sigma, 1e-12)
    return np.exp(-((x - mean) ** 2) / (2.0 * sigma ** 2))

# helper para extrair predição do retorno de saida.saida(...)
def extract_prediction(saida_ret):
    """
    saida.saida pode retornar:
      - array_like (predicoes)
      - tuple/list (ys, w, y_vec, b) ou (y_pred_array, ...)
    Aqui pegamos o primeiro elemento se é tupla/lista, senão usamos direto.
    """
    if isinstance(saida_ret, (list, tuple)):
        return np.asarray(saida_ret[0]).ravel()
    else:
        return np.asarray(saida_ret).ravel()

# helper para extrair escalar de objeto possivelmente 0-d ou array com 1 elemento
def scalar_of(x):
    arr = np.asarray(x).ravel()
    return float(arr[0])

In [21]:
for exec_id  in range(17, 21):
    print(f"Execução: ", {exec_id })
    # -------------------------
    # Inicialização de p e q
    # -------------------------
    rng = np.random.default_rng()
    p = rng.random((nin, NFP_INIT))
    q = rng.random(NFP_INIT)

    # -------------------------
    # Gera população inicial
    # cada indivíduo terá chaves: 'nfps', 'cs', 'ss', 'saida', 'fitness'
    # -------------------------
    pop = []
    for z in range(TAM_POP):
        nfpSort = NFP_INIT
        cs = np.empty((nin, nfpSort))
        ss = np.empty((nin, nfpSort))
        for j in range(nfpSort):
            for i in range(nin):
                cs[i, j] = xmin[i] + rng.random() * (xmax[i] - xmin[i])
                ss[i, j] = rng.random() * (xmax[i] - xmin[i])

        indiv = {'nfps': nfpSort, 'cs': cs, 'ss': ss}
        saida_full = saida.saida(xt, cs, ss, p, q, nfpSort)
        y_pred = extract_prediction(saida_full)
        indiv['saida'] = y_pred
        indiv['fitness'] = (0.5 * np.sum((indiv['saida'] - ydt) ** 2)) / npt
        pop.append(indiv)

    init_dump = {
        'pop': pop,
        'p': p.copy(),
        'q': q.copy(),
        'params': {
            'NFP_INIT': NFP_INIT, 'TAM_POP': TAM_POP, 'NUM_GERACOES': NUM_GERACOES,
            'TAXA_CRUZA': TAXA_CRUZA, 'TAXA_MUTA': TAXA_MUTA
        },
        'xmin': xmin, 'xmax': xmax,
        'rng_state': rng.bit_generator.state
    }
    with open(f'init_state_{exec_id }.pkl', 'wb') as f:
        pickle.dump(init_dump, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"-> Estado inicial salvo em init_state_{exec_id }.pkl (pickle)")

    # identifica melhor indivíduo inicial (índice)
    melhorindv = int(np.argmin([ind['fitness'] for ind in pop]))

    # inicializa parâmetros com o melhor indivíduo
    c = pop[melhorindv]['cs'].copy()
    s = pop[melhorindv]['ss'].copy()
    nfp = pop[melhorindv]['nfps']
    novapop = pop.copy()

    # -------------------------
    # Preparar xval para plot das MF (apenas para primeira feature)
    # -------------------------
    xval = np.linspace(xmin[0], xmax[0], npt)


    # -------------------------
    # Saída inicial sem treinamento
    # -------------------------
    yst = extract_prediction(saida.saida(xt, c, s, p, q, nfp))
    ysv = extract_prediction(saida.saida(xv, c, s, p, q, nfp))

    # -------------------------
    # Treinamento (laço de gerações + atualização p,q por época)
    # -------------------------
    erro = []
    erro2 = []
    y_train_pred = yst.copy()

    # inicializa early stopping
    best_error = (0.5 * np.sum((y_train_pred - ydt) ** 2)) / npt
    no_improve_count = 0

    for gen in range(NUM_GERACOES):
        if gen%50==0:
            print(f'Geração {gen+1}/{NUM_GERACOES}')
        # erro antes das atualizações p,q desta geração (apêndice para histórico)
        erro.append((0.5 * np.sum((y_train_pred - ydt) ** 2)) / npt)
        dyjdqj = 1.0

        # atualização estilo gradiente para p e q
        for _ in range(NEPOCA):
            for k in range(npt):
                sample = xt[k, :]
                ys_full = saida.saida(sample, c, s, p, q, nfp)
                ys = scalar_of(ys_full[0])
                w = np.asarray(ys_full[1]).ravel()
                y_vec = np.asarray(ys_full[2]).ravel()
                b = scalar_of(ys_full[3])

                dedys = float(ys) - float(ydt[k])

                for j in range(nfp):
                    dysdyj = w[j] / b
                    for i in range(nin):
                        dyjdpj = sample[i]
                        p[i, j] = p[i, j] - ((ALFA) * dedys * dysdyj * dyjdpj)
                    q[j] = q[j] - ((ALFA) * dedys * dysdyj * dyjdqj)

        # aplicar operador genético para gerar nova população
        pop = gerarnovapop.gerarnovapop(pop, melhorindv, TAM_POP, TAXA_CRUZA, TAXA_MUTA, xmax, xmin)

        for z in range(TAM_POP):
            # O step_size define o tamanho do ajuste fino
            pop[z] = buscalocal.busca_local_estocastica(
                pop[z], xt, ydt, p, q, pop[z]['nfps'], step_size=0.01
            )

        # re-avaliar fitness da nova população (sempre usando os pesos p,q atuais e dados de treino)
        for z in range(TAM_POP):
            indiv = pop[z]
            saida_full = saida.saida(xt, indiv['cs'], indiv['ss'], p, q, indiv['nfps'])
            y_pred = extract_prediction(saida_full)
            indiv['saida'] = y_pred
            indiv['fitness'] = (0.5 * np.sum((indiv['saida'] - ydt) ** 2)) / npt

        # atualizar melhor indivíduo
        melhorindv = int(np.argmin([ind['fitness'] for ind in pop]))

        # atualizar parâmetros a partir do melhor indivíduo
        c = pop[melhorindv]['cs'].copy()
        s = pop[melhorindv]['ss'].copy()
        nfp = pop[melhorindv]['nfps']

        # recomputar predição de treino com parâmetros atualizados
        y_train_pred = extract_prediction(saida.saida(xt, c, s, p, q, nfp))
        current_error = (0.5 * np.sum((y_train_pred - ydt) ** 2)) / npt
        erro2.append(current_error)


    # adiciona último erro (mantive sua lógica original para histórico)
    erro.append((0.5 * np.sum((y_train_pred - ydt) ** 2)) / npt)
    erro2.append((0.5 * np.sum((y_train_pred - ydt) ** 2)) / npt)

    # -------------------------
    # Predições finais (com os parâmetros finais)
    # -------------------------
    y_train_pred_final = extract_prediction(saida.saida(xt, c, s, p, q, nfp))
    y_val_pred_final = extract_prediction(saida.saida(xv, c, s, p, q, nfp))

    end_time = time.time()
    print(f"Tempo de execução: {end_time - start_time:.3f} s")

    c = np.array(c)
    s = np.array(s)

    # -------------------------
    # Métricas (validação e teste)
    # -------------------------
    mse_val = 0.5 * mean_squared_error(ydv, y_val_pred_final)
    rmse_val = np.sqrt(mse_val)
    r2_val = r2_score(ydv, y_val_pred_final)

    mse_train = 0.5 * mean_squared_error(ydt, y_train_pred_final)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(ydt, y_train_pred_final)

    print("\n===== MÉTRICAS =====")
    print(f"Treino   -> RMSE: {rmse_train:.6f}  R2: {r2_train:.6f}")
    print(f"Validação-> RMSE: {rmse_val:.6f}  R2: {r2_val:.6f}")

    final_dump = {
    'c_final': c.copy(),
    's_final': s.copy(),
    'p_final': p.copy(),
    'q_final': q.copy(),
    'nfp_final': nfp,

    # métricas finais
    'rmse_train': float(rmse_train),
    'r2_train': float(r2_train),
    'rmse_val': float(rmse_val),
    'r2_val': float(r2_val),

    # erros por geração
    'erro_hist_1': erro,
    'erro_hist_2': erro2,

    # dados usados
    'xt': xt,
    'ydt': ydt,
    'xv': xv,
    'ydv': ydv,

    # tempo e parâmetros do treinamento
    'tempo_total_s': end_time - start_time,
    'num_geracoes': NUM_GERACOES,
    'geracao_final': gen + 1,   # gen vem do loop
    'early_stop': (no_improve_count >= PATIENCE)
    }

    with open(f'final_state_{exec_id}.pkl', 'wb') as f:
        pickle.dump(final_dump, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"\n-> Estado final salvo em final_state_{exec_id}.pkl\n")


Execução:  {17}
-> Estado inicial salvo em init_state_17.pkl (pickle)
Geração 1/300
Geração 51/300
Geração 101/300
Geração 151/300
Geração 201/300
Geração 251/300
Tempo de execução: 845.943 s

===== MÉTRICAS =====
Treino   -> RMSE: 0.023553  R2: 0.898495
Validação-> RMSE: 0.028172  R2: 0.881716

-> Estado final salvo em final_state_17.pkl

Execução:  {18}
-> Estado inicial salvo em init_state_18.pkl (pickle)
Geração 1/300
Geração 51/300
Geração 101/300
Geração 151/300
Geração 201/300
Geração 251/300
Tempo de execução: 1655.762 s

===== MÉTRICAS =====
Treino   -> RMSE: 0.021858  R2: 0.912576
Validação-> RMSE: 0.027482  R2: 0.887445

-> Estado final salvo em final_state_18.pkl

Execução:  {19}
-> Estado inicial salvo em init_state_19.pkl (pickle)
Geração 1/300
Geração 51/300
Geração 101/300
Geração 151/300
Geração 201/300
Geração 251/300
Tempo de execução: 2494.208 s

===== MÉTRICAS =====
Treino   -> RMSE: 0.022627  R2: 0.906316
Validação-> RMSE: 0.027438  R2: 0.887806

-> Estado final s

In [24]:
rmse = []
#C:/Users/User/OneDrive/Área de Trabalho/backup/
for file in range(21):
    with open(f'final_state_{file}.pkl', 'rb') as arquivo:
                objeto_carregado = pickle.load(arquivo)
                rmse.append(objeto_carregado['rmse_val'])  
                
rmse = np.array(rmse)

In [25]:
print(np.array2string(rmse, formatter={'float_kind': lambda x: f"{x:.8f}".replace('.', ',')}))

[0,02718487 0,02999436 0,02837321 0,02801999 0,02736167 0,02801532
 0,02936621 0,02851321 0,03149331 0,02791824 0,02895972 0,02752639
 0,04281378 0,02638743 0,04531616 0,02746481 0,02803152 0,02817242
 0,02748163 0,02743754 0,02759582]


In [11]:
rmse.mean(), rmse.min(), rmse.max()

(np.float64(0.030161188555096072),
 np.float64(0.02638743168507536),
 np.float64(0.04531615794847611))